In [ ]:
import sys

print(sys.executable)
print(sys.version)

#!{sys.executable} -m pip install ysoisochrone

In [ ]:
# ============================================================
# TIC 138829403 — PROPIEDADES DOMINANTES DESDE MASSAGE
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

T_SUN = 5772.0
# Cambiar de TIC para cada estrella
tic_id = "TIC 138829403"

# ------------------------------------------------------------
# Localizar la estrella en el catálogo
# ------------------------------------------------------------

catalogo = pd.read_csv(
    "/home/luciam/Documents/starry/"
    "Resultados_Orion/catalogo_completo_mist_sommers.csv"
)

tic_num = int(tic_id.replace("TIC ", ""))

star = catalogo.loc[
    catalogo["TIC"] == tic_num
]

if len(star) == 0:
    raise ValueError(f"{tic_id} no encontrado en el catálogo.")

star = star.iloc[0]

# ------------------------------------------------------------
# MASSAGE MIST
# ------------------------------------------------------------

T_dom = float(star["T_median_1"])
sigma_T_dom = float(star["T_std_1"])

M_dom = float(star["mass_median_1"])
sigma_M_dom = float(star["mass_std_1"])

logL_dom = float(star["lgL_median_1"])
sigma_logL_dom = float(star["lgL_std_1"])

age_dom = float(star["age_median_1"])
sigma_age_dom = float(star["age_std_1"])

# ------------------------------------------------------------
# Radio derivado
# ------------------------------------------------------------

L_dom = 10**logL_dom

R_dom = (
    np.sqrt(L_dom)
    * (T_dom / T_SUN)**(-2)
)

sigma_R_dom = R_dom * np.sqrt(
    (0.5 * np.log(10) * sigma_logL_dom)**2
    +
    (2 * sigma_T_dom / T_dom)**2
)

print("=" * 70)
print(tic_id)
print("=" * 70)

print(f"T_dom       = {T_dom:.2f} ± {sigma_T_dom:.2f} K")
print(f"M_dom       = {M_dom:.4f} ± {sigma_M_dom:.4f} Msun")
print(f"log L_dom   = {logL_dom:.4f} ± {sigma_logL_dom:.4f}")
print(f"L_dom       = {L_dom:.4f} Lsun")
print(f"R_dom       = {R_dom:.4f} ± {sigma_R_dom:.4f} Rsun")
print(f"Age_dom     = {age_dom:.3f} ± {sigma_age_dom:.3f} Myr")

In [ ]:
# ============================================================
# CARGAR MIST Y SELECCIONAR PRE-MS
# ============================================================

path_modelos = Path(
    "/home/luciam/Documents/starry/"
    "Resultados_Orion/datos_estrellas_massage_orion/modelos"
)

mist = pd.read_csv(path_modelos / "MIST_V2.csv")

# Solamente fase joven / pre-MS
mist_pms = mist.loc[
    mist["phase"] == -1
].copy()

# ------------------------------------------------------------
# Convertir cantidades a unidades físicas
# ------------------------------------------------------------

mist_pms["Age_Myr"] = mist_pms["star_age"] / 1e6
mist_pms["Teff_K"] = 10**mist_pms["log_Teff"]
mist_pms["L_Lsun"] = 10**mist_pms["log_L"]
mist_pms["R_Rsun"] = 10**mist_pms["log_R"]

print("=" * 70)
print("MIST PRE-MS")
print("=" * 70)

print(f"Puntos originales : {len(mist):,}")
print(f"Puntos pre-MS     : {len(mist_pms):,}")

print(
    f"Edad: "
    f"{mist_pms['Age_Myr'].min():.5f} – "
    f"{mist_pms['Age_Myr'].max():.2f} Myr"
)

print(
    f"Masa: "
    f"{mist_pms['star_mass'].min():.3f} – "
    f"{mist_pms['star_mass'].max():.3f} Msun"
)

In [ ]:
# ============================================================
# AJUSTE MIST — COMPONENTE DOMINANTE
# ============================================================

fit = mist_pms.copy()

# ------------------------------------------------------------
# Chi-square
# ------------------------------------------------------------

fit["chi2_T"] = (
    (fit["Teff_K"] - T_dom) / sigma_T_dom
)**2

fit["chi2_L"] = (
    (fit["log_L"] - logL_dom) / sigma_logL_dom
)**2

fit["chi2_M"] = (
    (fit["star_mass"] - M_dom) / sigma_M_dom
)**2

fit["chi2"] = (
    fit["chi2_T"]
    + fit["chi2_L"]
    + fit["chi2_M"]
)

# ------------------------------------------------------------
# Mejor modelo
# ------------------------------------------------------------

best_idx = fit["chi2"].idxmin()
best = fit.loc[best_idx]

print("=" * 70)
print("MEJOR MODELO MIST — COMPONENTE DOMINANTE")
print("=" * 70)

print(f"Edad       = {best['Age_Myr']:.5f} Myr")
print(f"Masa       = {best['star_mass']:.6f} Msun")
print(f"Teff       = {best['Teff_K']:.2f} K")
print(f"L          = {best['L_Lsun']:.6f} Lsun")
print(f"R          = {best['R_Rsun']:.6f} Rsun")
print(f"log g      = {best['log_g']:.4f}")
print(f"log L      = {best['log_L']:.4f}")
print(f"chi2       = {best['chi2']:.4f}")

print()
print("Componentes de chi-square:")
print(f"  chi2_T = {best['chi2_T']:.4f}")
print(f"  chi2_L = {best['chi2_L']:.4f}")
print(f"  chi2_M = {best['chi2_M']:.4f}")

In [ ]:
# ============================================================
# REGIÓN CERCANA AL MEJOR AJUSTE
# ============================================================

delta_chi2_max = 20

region = fit.loc[
    fit["chi2"] <= best["chi2"] + delta_chi2_max
].copy()

print(f"Modelos dentro de Δχ² ≤ {delta_chi2_max}: {len(region):,}")

print("\nRango de masas:")
print(
    region["star_mass"].min(),
    "→",
    region["star_mass"].max()
)

print("\nRango de edades:")
print(
    region["Age_Myr"].min(),
    "→",
    region["Age_Myr"].max()
)

In [ ]:
# ============================================================
# EDAD ADOPTADA DEL SISTEMA — MASSAGE
# ============================================================

age_sys_Myr = float(star["age_median_1"])
sigma_age_sys_Myr = float(star["age_std_1"])

# MASSAGE: edad en Myr
# Isócronas: log10(edad / años)

logAge_sys = np.log10(age_sys_Myr * 1e6)

print("=" * 65)
print("EDAD ADOPTADA — MASSAGE")
print("=" * 65)
print(f"Edad MASSAGE = {age_sys_Myr:.4f} ± {sigma_age_sys_Myr:.4f} Myr")
print(f"logAge        = {logAge_sys:.5f}")
print("=" * 65)

In [ ]:
import inspect

from ysoisochrone import isochrone

print("isochrone:")
print(isochrone)

print("\nTipo:")
print(type(isochrone))

print("\nContenido:")
print(dir(isochrone))

In [ ]:
# ============================================================
# CARGAR ISOCRONAS BHAC15 VIA ysoisochrone
# ============================================================

import numpy as np
from scipy.interpolate import RegularGridInterpolator
from ysoisochrone.isochrone import Isochrone


# ------------------------------------------------------------
# 1. Inicializar Isochrone
# ------------------------------------------------------------

iso = Isochrone()

# Cargar modelo Baraffe et al. 2015 (BHAC15)
iso.set_tracks("Baraffe2015")


# ------------------------------------------------------------
# 2. Extraer la malla BHAC15
# ------------------------------------------------------------

log_age_grid = np.asarray(iso.log_age)
mass_grid    = np.asarray(iso.masses)

# logtlogl[:,:,0] = log10(Teff)
# logtlogl[:,:,1] = log10(L/Lsun)
logT_grid = np.asarray(iso.logtlogl[:, :, 0])
logL_grid = np.asarray(iso.logtlogl[:, :, 1])


# ------------------------------------------------------------
# 3. Comprobar dimensiones
# ------------------------------------------------------------

print("=" * 65)
print("MALLA BHAC15 — ysoisochrone")
print("=" * 65)

print(f"log_age_grid : {log_age_grid.shape}")
print(f"mass_grid    : {mass_grid.shape}")
print(f"logT_grid    : {logT_grid.shape}")
print(f"logL_grid    : {logL_grid.shape}")

print()
print(
    f"Rango de masas : "
    f"{mass_grid.min():.3f} - {mass_grid.max():.3f} Msun"
)

print(
    f"Rango de edad  : "
    f"{10**log_age_grid.min()/1e6:.4f} - "
    f"{10**log_age_grid.max()/1e6:.2f} Myr"
)


# ------------------------------------------------------------
# 4. Crear interpoladores
# ------------------------------------------------------------

interp_logT = RegularGridInterpolator(
    (log_age_grid, mass_grid),
    logT_grid,
    bounds_error=False,
    fill_value=np.nan
)

interp_logL = RegularGridInterpolator(
    (log_age_grid, mass_grid),
    logL_grid,
    bounds_error=False,
    fill_value=np.nan
)


print()
print("Interpoladores creados correctamente.")
print("=" * 65)

In [ ]:
print("=" * 65)
print("COMPROBACIÓN DE LA EDAD")
print("=" * 65)

print(f"Edad adoptada      = {age_sys_Myr:.3f} Myr")
print(f"logAge adoptado    = {logAge_sys:.5f}")
print(
    f"Rango isócronas    = "
    f"{log_age_grid.min():.5f} – "
    f"{log_age_grid.max():.5f}"
)

if log_age_grid.min() <= logAge_sys <= log_age_grid.max():
    print("✓ La edad está dentro del rango de las isócronas.")
else:
    print("⚠ La edad está fuera del rango de las isócronas (BHAC15 solo cubre hasta ~500 Myr).")

In [ ]:
# ============================================================
# PRUEBA DE INTERPOLACIÓN A LA EDAD DEL SISTEMA
# ============================================================

age_sys_Myr = float(star["age_median_1"])

logAge_sys = np.log10(age_sys_Myr * 1e6)

print(f"Edad MASSAGE = {age_sys_Myr:.4f} Myr")
print(f"logAge       = {logAge_sys:.5f}")


In [ ]:
# ============================================================
# RELACIÓN MASA–RADIO BHAC15 A LA EDAD DE MASSAGE
# ============================================================

# Edad adoptada
age_sys_Myr = float(star["age_median_1"])
logAge_sys = np.log10(age_sys_Myr * 1e6)

# Malla de masas
m_grid = np.linspace(
    mass_grid.min(),
    mass_grid.max(),
    2000
)

# Puntos para interpolar a edad fija
points = np.column_stack([
    np.full_like(m_grid, logAge_sys),
    m_grid
])

# Obtener log Teff y log luminosidad
logT_model = interp_logT(points)
logL_model = interp_logL(points)

# Convertir a unidades físicas
Teff_model = 10**logT_model
L_model = 10**logL_model

print("=" * 65)
print("BHAC15 A EDAD FIJA")
print("=" * 65)
print(f"Edad adoptada : {age_sys_Myr:.4f} Myr")
print(f"logAge        : {logAge_sys:.5f}")
print(f"Masa mínima   : {mass_grid.min():.3f} Msun")
print(f"Masa máxima   : {mass_grid.max():.3f} Msun")
print("=" * 65)

In [ ]:
# ============================================================
# CARGAR ISOCRONAS BHAC15 VIA ysoisochrone
# ============================================================

import numpy as np
from scipy.interpolate import RegularGridInterpolator
from ysoisochrone.isochrone import Isochrone


# ------------------------------------------------------------
# 1. Inicializar el objeto Isochrone
# ------------------------------------------------------------

iso = Isochrone()


# ------------------------------------------------------------
# 2. Cargar las isócronas Baraffe et al. (2015) / BHAC15
# ------------------------------------------------------------

iso.set_tracks("Baraffe2015")


# ------------------------------------------------------------
# 3. Extraer la malla
# ------------------------------------------------------------

log_age_grid = np.asarray(iso.log_age)
mass_grid    = np.asarray(iso.masses)

# logtlogl[:, :, 0] = log10(Teff)
# logtlogl[:, :, 1] = log10(L/Lsun)

logT_grid = np.asarray(iso.logtlogl[:, :, 0])
logL_grid = np.asarray(iso.logtlogl[:, :, 1])


# ------------------------------------------------------------
# 4. Comprobar dimensiones
# ------------------------------------------------------------

print("=" * 65)
print("MALLA BHAC15 — ysoisochrone")
print("=" * 65)

print(f"log_age_grid : {log_age_grid.shape}")
print(f"mass_grid    : {mass_grid.shape}")
print(f"logT_grid    : {logT_grid.shape}")
print(f"logL_grid    : {logL_grid.shape}")

print()

print(
    f"Rango de masas : "
    f"{mass_grid.min():.3f} - {mass_grid.max():.3f} Msun"
)

print(
    f"Rango de edad  : "
    f"{10**log_age_grid.min()/1e6:.4f} - "
    f"{10**log_age_grid.max()/1e6:.2f} Myr"
)


# ------------------------------------------------------------
# 5. Crear interpoladores
# ------------------------------------------------------------

interp_logT = RegularGridInterpolator(
    (log_age_grid, mass_grid),
    logT_grid,
    bounds_error=False,
    fill_value=np.nan
)

interp_logL = RegularGridInterpolator(
    (log_age_grid, mass_grid),
    logL_grid,
    bounds_error=False,
    fill_value=np.nan
)


print()
print("✓ Interpoladores creados correctamente")
print("=" * 65)

In [ ]:
# ============================================================
# EDAD ADOPTADA DEL SISTEMA
# ============================================================

age_sys_Myr = float(star["age_median_1"])

logAge_sys = np.log10(age_sys_Myr * 1e6)

print("=" * 65)
print("EDAD ADOPTADA")
print("=" * 65)
print(f"Edad MASSAGE = {age_sys_Myr:.4f} Myr")
print(f"logAge       = {logAge_sys:.5f}")
print("=" * 65)

In [ ]:
# ============================================================
# ISÓCRONA BHAC15 A LA EDAD ADOPTADA
# Masa – Radio – Teff – Luminosidad
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Malla de masas
# ------------------------------------------------------------

# Usamos únicamente el rango cubierto por BHAC15
m_grid = np.linspace(
    mass_grid.min(),
    mass_grid.max(),
    2000
)

# Edad fija para todo el sistema
points = np.column_stack([
    np.full_like(m_grid, logAge_sys),
    m_grid
])

# ------------------------------------------------------------
# 2. Interpolar propiedades BHAC15
# ------------------------------------------------------------

logT_model = interp_logT(points)
logL_model = interp_logL(points)

# Convertir a unidades físicas
Teff_model = 10.0**logT_model
L_model = 10.0**logL_model

# ------------------------------------------------------------
# 3. Calcular radio mediante Stefan–Boltzmann
#
# R/Rsun = sqrt(L/Lsun) * (Teff/Teff_sun)^(-2)
# ------------------------------------------------------------

T_sun = 5772.0  # K

R_model = (
    np.sqrt(L_model)
    * (Teff_model / T_sun)**(-2)
)

# ------------------------------------------------------------
# 4. Eliminar puntos fuera de la región válida
# ------------------------------------------------------------

valid = (
    np.isfinite(Teff_model)
    & np.isfinite(L_model)
    & np.isfinite(R_model)
)

m_iso = m_grid[valid]
Teff_iso = Teff_model[valid]
L_iso = L_model[valid]
logL_iso = logL_model[valid]
R_iso = R_model[valid]

# ------------------------------------------------------------
# 5. Crear DataFrame de la isócrona
# ------------------------------------------------------------

iso_BHAC15_age = pd.DataFrame({
    "Mass": m_iso,
    "Teff": Teff_iso,
    "logL": logL_iso,
    "Luminosity": L_iso,
    "Radius": R_iso
})

print("=" * 70)
print("ISÓCRONA BHAC15")
print("=" * 70)
print(f"Edad adoptada : {age_sys_Myr:.4f} Myr")
print(f"log(age/yr)   : {logAge_sys:.5f}")
print()
print(f"Masa mínima   : {m_iso.min():.4f} Msun")
print(f"Masa máxima   : {m_iso.max():.4f} Msun")
print(f"N puntos      : {len(m_iso)}")
print("=" * 70)

display(iso_BHAC15_age.head())

In [ ]:
# ============================================================
# BHAC15 — ISÓCRONA A LA EDAD ADOPTADA
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Edad adoptada
# ------------------------------------------------------------

age_sys_Myr = float(star["age_median_1"])
logAge_sys = np.log10(age_sys_Myr * 1e6)

# ------------------------------------------------------------
# 2. Malla de masas
# ------------------------------------------------------------

m_grid = np.linspace(
    float(np.nanmin(mass_grid)),
    float(np.nanmax(mass_grid)),
    2000
)

# ------------------------------------------------------------
# 3. Puntos de interpolación
#
# Cada punto es:
# (logAge, Mass)
# ------------------------------------------------------------

points = np.column_stack([
    np.full(len(m_grid), logAge_sys, dtype=float),
    m_grid
])

# ------------------------------------------------------------
# 4. Interpolar BHAC15
# ------------------------------------------------------------

logT_model = np.asarray(
    interp_logT(points),
    dtype=float
)

logL_model = np.asarray(
    interp_logL(points),
    dtype=float
)

# ------------------------------------------------------------
# 5. Convertir a unidades físicas
# ------------------------------------------------------------

Teff_model = 10.0**logT_model
L_model = 10.0**logL_model

# ------------------------------------------------------------
# 6. Calcular radio
#
# R/Rsun = sqrt(L/Lsun) *
#          (Teff/Teff_sun)^(-2)
# ------------------------------------------------------------

T_sun = 5772.0

R_model = (
    np.sqrt(L_model)
    * (Teff_model / T_sun)**(-2)
)

# ------------------------------------------------------------
# 7. Seleccionar puntos válidos
# ------------------------------------------------------------

valid = (
    np.isfinite(m_grid)
    & np.isfinite(logT_model)
    & np.isfinite(logL_model)
    & np.isfinite(Teff_model)
    & np.isfinite(L_model)
    & np.isfinite(R_model)
)

m_iso = m_grid[valid]
Teff_iso = Teff_model[valid]
logL_iso = logL_model[valid]
L_iso = L_model[valid]
R_iso = R_model[valid]

# ------------------------------------------------------------
# 8. Guardar la isócrona en un DataFrame
# ------------------------------------------------------------

iso_BHAC15_age = pd.DataFrame({
    "Mass": m_iso,
    "Teff": Teff_iso,
    "logL": logL_iso,
    "Luminosity": L_iso,
    "Radius": R_iso
})

# ------------------------------------------------------------
# 9. Resumen
# ------------------------------------------------------------

print("=" * 70)
print("BHAC15 — ISÓCRONA A LA EDAD ADOPTADA")
print("=" * 70)

print(f"Edad adoptada : {age_sys_Myr:.4f} Myr")
print(f"log(age/yr)   : {logAge_sys:.5f}")
print()

print(
    f"Masa          : "
    f"{m_iso.min():.4f} – {m_iso.max():.4f} Msun"
)

print(
    f"Teff          : "
    f"{Teff_iso.min():.1f} – {Teff_iso.max():.1f} K"
)

print(
    f"Radio         : "
    f"{R_iso.min():.4f} – {R_iso.max():.4f} Rsun"
)

print(
    f"logL          : "
    f"{logL_iso.min():.4f} – {logL_iso.max():.4f}"
)

print()
print(f"Puntos válidos: {len(m_iso)}")
print("=" * 70)

display(iso_BHAC15_age.head(10))

In [ ]:
# ============================================================
# BHAC15 — MASA vs RADIO
# ============================================================

fig, ax = plt.subplots(figsize=(7, 5.5))

ax.plot(
    m_iso,
    R_iso,
    lw=2.2
)

ax.set_xlabel(
    r"$M\ [M_\odot]$",
    fontsize=13
)

ax.set_ylabel(
    r"$R\ [R_\odot]$",
    fontsize=13
)

ax.set_title(
    fr"BHAC15 — Masa–Radio "
    fr"($t={age_sys_Myr:.2f}$ Myr)",
    fontsize=14
)

ax.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BHAC15 — MASA vs TEMPERATURA EFECTIVA
# ============================================================

fig, ax = plt.subplots(figsize=(7, 5.5))

ax.plot(
    m_iso,
    Teff_iso,
    lw=2.2
)

ax.set_xlabel(
    r"$M\ [M_\odot]$",
    fontsize=13
)

ax.set_ylabel(
    r"$T_{\rm eff}\ [K]$",
    fontsize=13
)

ax.set_title(
    fr"BHAC15 — Masa–$T_{{\rm eff}}$ "
    fr"($t={age_sys_Myr:.2f}$ Myr)",
    fontsize=14
)

ax.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BHAC15 — MASA vs LUMINOSIDAD
# ============================================================

fig, ax = plt.subplots(figsize=(7, 5.5))

ax.plot(
    m_iso,
    logL_iso,
    lw=2.2
)

ax.set_xlabel(
    r"$M\ [M_\odot]$",
    fontsize=13
)

ax.set_ylabel(
    r"$\log(L/L_\odot)$",
    fontsize=13
)

ax.set_title(
    fr"BHAC15 — Masa–Luminosidad "
    fr"($t={age_sys_Myr:.2f}$ Myr)",
    fontsize=14
)

ax.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# COMPROBACIÓN DE CONSISTENCIA
# ============================================================

R_check = (
    np.sqrt(iso_BHAC15_age["Luminosity"].values)
    * (
        iso_BHAC15_age["Teff"].values / T_sun
    )**(-2)
)

difference = (
    R_check - iso_BHAC15_age["Radius"].values
)

print("=" * 70)
print("COMPROBACIÓN R(L, Teff)")
print("=" * 70)

print(
    f"Diferencia máxima = "
    f"{np.nanmax(np.abs(difference)):.7e} Rsun"
)

print(
    f"Diferencia media  = "
    f"{np.nanmean(np.abs(difference)):.7e} Rsun"
)

print("=" * 70)

In [ ]:
# ============================================================
# PROPIEDADES DEL COMPONENTE DOMINANTE — MASSAGE / MIST
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Parámetros de MASSAGE / MIST
# ------------------------------------------------------------

T_dom_MIST = float(star["T_median_1"])
sigma_T_dom_MIST = float(star["T_std_1"])

M_dom_MIST = float(star["mass_median_1"])
sigma_M_dom_MIST = float(star["mass_std_1"])

age_dom_MIST = float(star["age_median_1"])
sigma_age_dom_MIST = float(star["age_std_1"])

logL_dom_MIST = float(star["lgL_median_1"])
sigma_logL_dom_MIST = float(star["lgL_std_1"])

L_dom_MIST = 10**logL_dom_MIST

# ------------------------------------------------------------
# Radio derivado de L y Teff
# ------------------------------------------------------------

T_sun = 5772.0

R_dom_MIST = (
    np.sqrt(L_dom_MIST)
    * (T_dom_MIST / T_sun)**(-2)
)

# Propagación de incertidumbre
frac_sigma_R_MIST = np.sqrt(
    (0.5 * np.log(10) * sigma_logL_dom_MIST)**2
    +
    (2.0 * sigma_T_dom_MIST / T_dom_MIST)**2
)

sigma_R_dom_MIST = R_dom_MIST * frac_sigma_R_MIST

# ------------------------------------------------------------
# Mostrar
# ------------------------------------------------------------

print("=" * 70)
print("COMPONENTE DOMINANTE — MASSAGE / MIST")
print("=" * 70)

print(f"Teff  = {T_dom_MIST:.2f} ± {sigma_T_dom_MIST:.2f} K")
print(f"Mass  = {M_dom_MIST:.4f} ± {sigma_M_dom_MIST:.4f} Msun")
print(f"logL  = {logL_dom_MIST:.5f} ± {sigma_logL_dom_MIST:.5f}")
print(f"L     = {L_dom_MIST:.5f} Lsun")
print(f"Age   = {age_dom_MIST:.4f} ± {sigma_age_dom_MIST:.4f} Myr")
print(f"Radius= {R_dom_MIST:.5f} ± {sigma_R_dom_MIST:.5f} Rsun")

print("=" * 70)

In [ ]:
# ============================================================
# BHAC15 A LA EDAD ADOPTADA
# ============================================================

age_sys_Myr = float(star["age_median_1"])
logAge_sys = np.log10(age_sys_Myr * 1e6)

print("=" * 70)
print("ISOCRONA BHAC15")
print("=" * 70)

print(f"Edad adoptada = {age_sys_Myr:.4f} Myr")
print(f"log(age/yr)   = {logAge_sys:.5f}")

print("=" * 70)

# ------------------------------------------------------------
# Copia limpia
# ------------------------------------------------------------

iso_age = iso_BHAC15_age.copy()

# Eliminar filas con datos inválidos
iso_age = iso_age.replace([np.inf, -np.inf], np.nan)

iso_age = iso_age.dropna(
    subset=["Mass", "Teff", "logL", "Radius"]
).copy()

iso_age = iso_age.sort_values("Mass").reset_index(drop=True)

print(f"Número de puntos BHAC15 = {len(iso_age)}")

display(iso_age.head())

In [ ]:
# ============================================================
# AJUSTE BHAC15 — MASA DEL COMPONENTE DOMINANTE
# ============================================================

# ------------------------------------------------------------
# Valores observacionales
# ------------------------------------------------------------

T_obs = T_dom_MIST
sigma_T_obs = sigma_T_dom_MIST

logL_obs = logL_dom_MIST
sigma_logL_obs = sigma_logL_dom_MIST

# ------------------------------------------------------------
# Residuales
# ------------------------------------------------------------

iso_age["res_T"] = (
    (iso_age["Teff"] - T_obs)
    / sigma_T_obs
)

iso_age["res_logL"] = (
    (iso_age["logL"] - logL_obs)
    / sigma_logL_obs
)

# ------------------------------------------------------------
# Chi-square total
# ------------------------------------------------------------

iso_age["chi2_T"] = iso_age["res_T"]**2
iso_age["chi2_logL"] = iso_age["res_logL"]**2

iso_age["chi2_total"] = (
    iso_age["chi2_T"]
    +
    iso_age["chi2_logL"]
)

# ------------------------------------------------------------
# Mejor modelo
# ------------------------------------------------------------

idx_best = iso_age["chi2_total"].idxmin()

best = iso_age.loc[idx_best]

M_BHAC15_best = float(best["Mass"])
T_BHAC15_best = float(best["Teff"])
logL_BHAC15_best = float(best["logL"])
L_BHAC15_best = float(best["Luminosity"])
R_BHAC15_best = float(best["Radius"])

chi2_min = float(best["chi2_total"])

# ------------------------------------------------------------
# Mostrar
# ------------------------------------------------------------

print("=" * 70)
print("MEJOR MODELO BHAC15")
print("=" * 70)

print(f"Masa       = {M_BHAC15_best:.5f} Msun")
print(f"Teff       = {T_BHAC15_best:.2f} K")
print(f"logL       = {logL_BHAC15_best:.5f}")
print(f"L          = {L_BHAC15_best:.5f} Lsun")
print(f"Radio      = {R_BHAC15_best:.5f} Rsun")
print(f"chi²       = {chi2_min:.5f}")

print("=" * 70)

In [ ]:
# ============================================================
# COMPARACIÓN MASSAGE vs BHAC15
# ============================================================

print("=" * 70)
print("MASSAGE / MIST  vs  BHAC15")
print("=" * 70)

print(f"{'Parámetro':<15} {'MASSAGE':>20} {'BHAC15':>20}")
print("-" * 70)

print(
    f"{'Masa (Msun)':<15} "
    f"{M_dom_MIST:>9.4f} ± {sigma_M_dom_MIST:<7.4f} "
    f"{M_BHAC15_best:>20.4f}"
)

print(
    f"{'Teff (K)':<15} "
    f"{T_dom_MIST:>9.1f} ± {sigma_T_dom_MIST:<7.1f} "
    f"{T_BHAC15_best:>20.1f}"
)

print(
    f"{'logL':<15} "
    f"{logL_dom_MIST:>9.4f} ± {sigma_logL_dom_MIST:<7.4f} "
    f"{logL_BHAC15_best:>20.4f}"
)

print(
    f"{'Radius (Rsun)':<15} "
    f"{R_dom_MIST:>9.4f} ± {sigma_R_dom_MIST:<7.4f} "
    f"{R_BHAC15_best:>20.4f}"
)

print("=" * 70)

In [ ]:
# ============================================================
# TENSION ENTRE MASSAGE Y BHAC15
# ============================================================

delta_T = T_BHAC15_best - T_dom_MIST
delta_logL = logL_BHAC15_best - logL_dom_MIST
delta_M = M_BHAC15_best - M_dom_MIST
delta_R = R_BHAC15_best - R_dom_MIST

z_T = delta_T / sigma_T_dom_MIST
z_logL = delta_logL / sigma_logL_dom_MIST
z_M = delta_M / sigma_M_dom_MIST
z_R = delta_R / sigma_R_dom_MIST

print("=" * 70)
print("COMPATIBILIDAD")
print("=" * 70)

print(f"ΔTeff   = {delta_T:+.2f} K   ({z_T:+.2f} σ)")
print(f"ΔlogL   = {delta_logL:+.5f}      ({z_logL:+.2f} σ)")
print(f"ΔM      = {delta_M:+.5f} Msun ({z_M:+.2f} σ)")
print(f"ΔR      = {delta_R:+.5f} Rsun ({z_R:+.2f} σ)")

print()
print(f"χ² mínimo = {chi2_min:.5f}")

print("=" * 70)

In [ ]:
# ============================================================
# INSPECCIÓN COMPLETA DEL SISTEMA
# ============================================================

print("=" * 80)
print("SISTEMA:", star.get("TIC", star.get("tic", "TIC desconocido")))
print("=" * 80)

for key, value in star.items():
    if any(x in str(key).lower() for x in [
        "mass", "temp", "teff", "age", "lgL", "lum", "radius"
    ]):
        print(f"{key:25s} : {value}")

print("=" * 80)

In [ ]:
star = catalogo.loc[
    catalogo["TIC"].astype(str) == "138829403"
].iloc[0]

cols = [
    "T_median_1",
    "T_std_1",
    "lgL_median_1",
    "lgL_std_1",
    "mass_median_1",
    "mass_std_1",
    "age_median_1",
    "age_std_1"
]

print(star[cols])

In [ ]:
T = float(star["T_median_1"])
sigma_T = float(star["T_std_1"])

logL = float(star["lgL_median_1"])
sigma_logL = float(star["lgL_std_1"])

L = 10**logL

R = np.sqrt(L) * (T / 5772.0)**(-2)

frac_sigma_R = np.sqrt(
    (0.5 * np.log(10) * sigma_logL)**2 +
    (2 * sigma_T / T)**2
)

sigma_R = R * frac_sigma_R

print(f"T_eff = {T:.1f} ± {sigma_T:.1f} K")
print(f"log L = {logL:.3f} ± {sigma_logL:.3f}")
print(f"L = {L:.2f} L_sun")
print(f"R = {R:.2f} ± {sigma_R:.2f} R_sun")